# Advanced Problems with Solutions: Python Optimizations — String Interning

This notebook contains advanced exercises on string interning, identity, equality, compiler behavior, `sys.intern`, benchmarking, and safe optimization practices.

> Important: string interning behavior is partly implementation-specific. These examples target CPython 3.11.

In [1]:
import sys
import dis
import timeit
import platform

print(sys.version)
print(platform.python_implementation())

3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]
CPython


## Problem 1 — Identity vs equality

Create two equal strings dynamically. Show that they are equal by value but not necessarily identical.

### Solution

In [2]:
a = ''.join(['hello', ' ', 'world'])
b = ''.join(['hello', ' ', 'world'])

print('a == b:', a == b)
print('a is b:', a is b)
print('id(a):', id(a))
print('id(b):', id(b))

a == b: True
a is b: False
id(a): 2313470769264
id(b): 2313470761456


`==` compares values. `is` compares object identity.

For normal string value comparison, use `==`, not `is`.

## Problem 2 — Identifier-like strings

Investigate which string literals are automatically interned.

### Solution

In [3]:
examples = [
    'hello',
    'hello_world',
    '_private_name',
    'name123',
    '123_name',
    'hello world',
    'hello-world',
    'hello, world!',
]

for s in examples:
    a = s
    b = s
    print(f'{s!r:20} a is b -> {a is b}')

'hello'              a is b -> True
'hello_world'        a is b -> True
'_private_name'      a is b -> True
'name123'            a is b -> True
'123_name'           a is b -> True
'hello world'        a is b -> True
'hello-world'        a is b -> True
'hello, world!'      a is b -> True


Many identifier-like strings are interned automatically by CPython.

However, this behavior should not be used for program logic.

## Problem 3 — Runtime strings are different from literals

Show that two strings with the same value can behave differently depending on whether they are literals or created at runtime.

### Solution

In [4]:
literal_a = 'python_identifier'
literal_b = 'python_identifier'

runtime_a = ''.join(['python', '_', 'identifier'])
runtime_b = ''.join(['python', '_', 'identifier'])

print('literal_a == literal_b:', literal_a == literal_b)
print('literal_a is literal_b:', literal_a is literal_b)

print('runtime_a == runtime_b:', runtime_a == runtime_b)
print('runtime_a is runtime_b:', runtime_a is runtime_b)

literal_a == literal_b: True
literal_a is literal_b: True
runtime_a == runtime_b: True
runtime_a is runtime_b: False


Interning is affected by how the string is created.

A string literal may be interned, while an equal runtime-created string may not be.

## Problem 4 — Force interning with `sys.intern`

Create two equal runtime strings, intern them, and compare identity before and after.

### Solution

In [5]:
x = ''.join(['data', '_', 'pipeline'])
y = ''.join(['data', '_', 'pipeline'])

print('Before interning:')
print('x == y:', x == y)
print('x is y:', x is y)

x_i = sys.intern(x)
y_i = sys.intern(y)

print('\nAfter interning:')
print('x_i == y_i:', x_i == y_i)
print('x_i is y_i:', x_i is y_i)

Before interning:
x == y: True
x is y: False

After interning:
x_i == y_i: True
x_i is y_i: True


`sys.intern` stores one canonical object for a string value.

After interning, equal strings can share identity.

## Problem 5 — Inspect compiler behavior with `dis`

Disassemble a function containing repeated string literals.

Explain why two variables may point to the same string object.

### Solution

In [6]:
def literal_demo():
    a = 'hello, world!'
    b = 'hello, world!'
    return a is b

print(literal_demo())
print(literal_demo.__code__.co_consts)
dis.dis(literal_demo)

True
(None, 'hello, world!')
  1           RESUME                   0

  2           LOAD_CONST               1 ('hello, world!')
              STORE_FAST               0 (a)

  3           LOAD_CONST               1 ('hello, world!')
              STORE_FAST               1 (b)

  4           LOAD_FAST_LOAD_FAST      1 (a, b)
              IS_OP                    0
              RETURN_VALUE


Even strings that are not identifier-like may be reused inside the same code object.

This is constant reuse, not something your code should rely on.

## Problem 6 — Fix a dangerous bug

The function below is wrong:

```python
def is_command_start(command):
    return command is 'START'
```

Fix it and prove the fix works.

### Solution

In [8]:
def bad_is_command_start(command: str) -> bool:
    return command is 'START'

def good_is_command_start(command: str) -> bool:
    return command == 'START'

literal_command = 'START'
runtime_command = ''.join(['S', 'T', 'A', 'R', 'T'])

print('bad literal:', bad_is_command_start(literal_command))
print('bad runtime:', bad_is_command_start(runtime_command))
print('good literal:', good_is_command_start(literal_command))
print('good runtime:', good_is_command_start(runtime_command))

assert good_is_command_start(literal_command)
assert good_is_command_start(runtime_command)

bad literal: True
bad runtime: False
good literal: True
good runtime: True


Use `==` for string values.

`is` can appear to work with interned strings, which makes the bug harder to notice.

## Problem 7 — Intern repeated symbols in a dataset

Build a list of many repeated runtime-created strings. Compare the number of unique object identities before and after interning.

### Solution

In [9]:
symbols = ['LOGIN', 'LOGOUT', 'PURCHASE', 'VIEW', 'CLICK'] * 20_000

raw = [''.join([s]) for s in symbols]
interned = [sys.intern(s) for s in raw]

print('unique values:', len(set(symbols)))
print('unique raw ids:', len({id(s) for s in raw}))
print('unique interned ids:', len({id(s) for s in interned}))

unique values: 5
unique raw ids: 5
unique interned ids: 5


Interning can reduce duplicate string objects when many repeated symbolic strings are kept in memory.

## Problem 8 — Benchmark equality before and after interning

Compare equality checks for long strings before and after interning.

### Solution

In [10]:
left = 'x' * 10_000 + 'END'
right = ''.join(['x' * 10_000, 'END'])

left_i = sys.intern(left)
right_i = sys.intern(right)

print('left == right:', left == right)
print('left is right:', left is right)
print('left_i is right_i:', left_i is right_i)

t1 = timeit.timeit('left == right', globals=globals(), number=500_000)
t2 = timeit.timeit('left_i == right_i', globals=globals(), number=500_000)

print(f'non-interned equality: {t1:.6f}')
print(f'interned equality:     {t2:.6f}')
print(f'ratio:                 {t1 / t2:.2f}x')

left == right: True
left is right: False
left_i is right_i: True
non-interned equality: 0.274761
interned equality:     0.011348
ratio:                 24.21x


Interned strings can make repeated equality checks faster because identity can short-circuit comparison.

However, optimize only after measuring real workloads.

## Problem 9 — Benchmark the wrong way and the better way

Explain why comparing literals can produce misleading benchmark results. Then benchmark dynamically-created strings.

### Solution

In [12]:
bad = timeit.timeit("'hello_world' is 'hello_world'", number=1_000_000)

better = timeit.timeit(
    "''.join(['hello', '_', 'world']) is ''.join(['hello', '_', 'world'])",
    number=1_000_000,
)

print('literal identity benchmark:', bad)
print('runtime identity benchmark:', better)

literal identity benchmark: 0.03660979960113764
runtime identity benchmark: 0.243711500428617


Literal benchmarks may measure compiler behavior and constant reuse.

Runtime construction gives a better view of whether separate objects are created.

## Problem 10 — Implement a small tokenizer using interning

Write a tokenizer that interns repeated tokens. Demonstrate that repeated token values share identity.

### Solution

In [13]:
def tokenize_and_intern(text: str) -> list[str]:
    return [sys.intern(token) for token in text.split()]

text = 'LOAD name STORE name LOAD value STORE value LOAD name'
tokens = tokenize_and_intern(text)

for token in sorted(set(tokens)):
    ids = {id(t) for t in tokens if t == token}
    print(f'{token:10} unique ids: {len(ids)}')

assert len({id(t) for t in tokens if t == 'name'}) == 1

LOAD       unique ids: 1
STORE      unique ids: 1
name       unique ids: 1
value      unique ids: 1


This is a realistic use case for interning: parsers, compilers, interpreters, and symbolic systems often reuse many repeated token strings.

## Problem 11 — Understand the memory trade-off

Interning can save memory for repeated strings, but it can be harmful for many unique strings.

Create many unique strings and intern them. Explain why this may be a bad idea.

### Solution

In [14]:
unique_strings = [f'user_generated_value_{i}' for i in range(10_000)]
interned_unique_strings = [sys.intern(s) for s in unique_strings]

print('unique values:', len(set(unique_strings)))
print('unique interned ids:', len({id(s) for s in interned_unique_strings}))

unique values: 10000
unique interned ids: 10000


Interning highly unique strings provides little or no deduplication benefit.

It may also keep strings alive longer than expected, increasing memory pressure.

## Problem 12 — Final classification challenge

Classify each comparison as safe or unsafe for production logic.

### Solution

In [16]:
cases = {
    "'abc' == ''.join(['a', 'b', 'c'])": 'abc' == ''.join(['a', 'b', 'c']),
    "'abc' is ''.join(['a', 'b', 'c'])": 'abc' is ''.join(['a', 'b', 'c']),
    "sys.intern('abc') is sys.intern(runtime abc)": sys.intern('abc') is sys.intern(''.join(['a', 'b', 'c'])),
    "None is None": None is None,
}

for expression, result in cases.items():
    print(f'{expression:55} -> {result}')

'abc' == ''.join(['a', 'b', 'c'])                       -> True
'abc' is ''.join(['a', 'b', 'c'])                       -> False
sys.intern('abc') is sys.intern(runtime abc)            -> True
None is None                                            -> True


| Expression type | Safe for value logic? | Explanation |
|---|---:|---|
| String `==` | Yes | Correct value comparison |
| String `is` | No | Depends on interning and object identity |
| `sys.intern(a) is sys.intern(b)` | Sometimes | Safe only if intentional canonical identity is the goal |
| `None is None` | Yes | `None` is a singleton |

## Final Takeaways

1. Use `==` for string equality.
2. Use `is` only for identity checks.
3. CPython automatically interns some strings, especially identifier-like strings.
4. `sys.intern` can manually canonicalize strings.
5. Interning is useful for repeated symbolic strings.
6. Interning many unique strings can waste memory.
7. Always benchmark real workloads before optimizing.